<!-- ============================================================ -->
<!-- NOTEBOOK HEADER — MLOps Introductory Course on GCP           -->
<!-- ============================================================ -->

<div style="border-bottom: 3px solid #4285F4; padding-bottom: 12px; margin-bottom: 20px;">

<div style="display: flex; align-items: center; justify-content: space-between;">
  <div>
    <img src="https://www.isae-supaero.fr/wp-content/uploads/2025/03/logo.svg" width="180">
  </div>
  <div style="text-align: right;">
    <img src="https://user-images.githubusercontent.com/63151412/167391313-4683cc69-2bf6-4597-b767-5c18e2bbbfa0.png" width="180">
  </div>
</div>

# Lab 04a — Kubeflow Pipelines Fundamentals

**Course:** MLOps Introductory Course on GCP · M2 Data Science · ISAE-SUPAERO  
**Lab created by:** Headmind Partners AI & Blockchain  
**Estimated duration:** ~1h00

</div>

## 📋 Lab Overview

In the previous labs you trained models, tracked experiments, and deployed endpoints — all as individual steps. In production, these steps must be **orchestrated** into reproducible, automated **ML pipelines**. **Kubeflow Pipelines (KFP)** is a framework-agnostic pipeline SDK that integrates natively with **Vertex AI Pipelines**, Google Cloud's fully managed pipeline orchestrator.

### Learning Objectives

1. Understand the core concepts of Kubeflow Pipelines: **components**, **pipelines**, and **artifacts**.
2. Build **lightweight Python components** using the `@dsl.component` decorator.
3. Compose components into a **pipeline DAG** and **compile** it to YAML.
4. Submit and monitor a pipeline run on **Vertex AI Pipelines**.
5. **Parameterize** a pipeline and re-run it with different hyperparameters.
6. Compare pipeline runs in the Vertex AI console.

### Notebook Structure

| # | Section | Focus |
|---|---------|-------|
| 0 | Setup | Install KFP SDK, imports, GCP configuration |
| 1 | KFP Concepts | Components, pipelines, artifacts, and the DAG model |
| 2 | Building Components | Create data-prep, training, and evaluation components |
| 3 | Compose & Compile | Wire components into a pipeline and compile to YAML |
| 4 | Run on Vertex AI | Submit the pipeline and inspect the run in the console |
| 5 | Parameterize & Re-run | Re-run with different hyperparameters and compare |
| 6 | Cleanup | Delete pipeline runs |

### How to Read This Notebook

- **`# TODO`** — Code you need to write. Look for the `######` delimiters.
- **`✏️ Question`** — A conceptual question. Write your answer in the markdown cell below it.
- Cells **without** a TODO are provided — read them, run them, and make sure you understand them.
- Documentation links are provided in 📖 callouts whenever a new API is introduced.

---
## 0 · Setup

### 0.1 Install dependencies

In [ ]:
%pip install --upgrade --quiet kfp google-cloud-aiplatform google-cloud-pipeline-components

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.4/418.4 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 39.8 MB/s eta 0:00:00


### 0.2 Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from kfp import dsl
from kfp.dsl import Input, Output, Dataset, Model, Metrics
from kfp import compiler

import google.cloud.aiplatform as aiplatform

import kfp
print(f"KFP SDK version: {kfp.__version__}")
print(f"Vertex AI SDK version: {aiplatform.__version__}")

KFP SDK version: 2.16.0
Vertex AI SDK version: 1.140.0


### 0.3 Configuration

Replace the placeholders below with your own GCP project details.

In [ ]:
# ── Constants ──
PROJECT_ID = "isae-sdd-481407"         # @param {type:"string"}
LOCATION = "europe-west3"               # @param {type:"string"}
BUCKET_URI = "gs://bucket-lab04"     # @param {type:"string"}

##############################  TODO  ##############################
# Set your_name to a unique lowercase identifier (e.g. first letter of first name + last name).
your_name = "mregaieg"  # TODO
####################################################################

# Pipeline artifacts will be stored under this root
PIPELINE_ROOT = f"{BUCKET_URI}/pipeline-root/lab04a/{your_name}"

# Initialize the Vertex AI SDK
aiplatform.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=BUCKET_URI,
)
print(f"✅ Vertex AI initialized — project={PROJECT_ID}, location={LOCATION}")
print(f"   Pipeline root: {PIPELINE_ROOT}")

✅ Vertex AI initialized — project=isae-sdd-481407, location=europe-west3
   Pipeline root: gs://bucket-lab04/pipeline-root/lab04a/mregaieg


> 📖 **Docs:** [`aiplatform.init()`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform#google_cloud_aiplatform_init)

In [ ]:
PIPELINE_DISPLAY_NAME = f"lab04a-covertype-{your_name}"
print(f"Pipeline display name: {PIPELINE_DISPLAY_NAME}")

Pipeline display name: lab04a-covertype-mregaieg


---
## 1 · Kubeflow Pipelines — Key Concepts

Before writing code, let's understand the three core abstractions of the KFP SDK:

**Component** — A self-contained unit of work (a Python function or a container image). Each component receives typed inputs and produces typed outputs. Think of it as a single step in your ML workflow: *load data*, *train model*, *evaluate*, etc.

**Pipeline** — A **directed acyclic graph (DAG)** of components. The pipeline defines the execution order by wiring outputs of one component to inputs of the next. KFP infers the DAG from these data dependencies.

**Artifact** — A typed object (dataset, model, metrics) that flows between components. KFP handles serialization and storage in Cloud Storage automatically.

The typical workflow is:

1. **Define** components with `@dsl.component`
2. **Compose** them into a pipeline with `@dsl.pipeline`
3. **Compile** the pipeline to a YAML file with `compiler.Compiler().compile()`
4. **Run** the YAML on Vertex AI Pipelines with `aiplatform.PipelineJob`

> 📖 **Docs:**
> - [KFP SDK v2 overview](https://www.kubeflow.org/docs/components/pipelines/user-guides/components/)
> - [Vertex AI Pipelines introduction](https://cloud.google.com/vertex-ai/docs/pipelines/introduction)

---
## 2 · Building Pipeline Components

We will build three components for a simple ML pipeline using the **Covertype** dataset (multi-class classification of forest cover types from cartographic features):

1. **`load_and_split_data`** — Fetch the dataset, subsample it, and split into train/test sets.
2. **`train_model`** — Train a scikit-learn classifier on the training set.
3. **`evaluate_model`** — Evaluate the model on the test set and log metrics.

Each component is a regular Python function decorated with `@dsl.component`. The decorator specifies the base Docker image and any pip packages the function needs.

> 📖 **Docs:** [`@dsl.component` decorator](https://www.kubeflow.org/docs/components/pipelines/user-guides/components/lightweight-python-components/)

### 2.1 Data preparation component

This component is **provided**. Read it carefully — it demonstrates:
- How to declare typed outputs (`Output[Dataset]`)
- How imports go inside the function
- How to write data to artifact paths

In [ ]:
@dsl.component(
    base_image="python:3.10",
    packages_to_install=["scikit-learn==1.5.2", "pandas==2.2.3"],
)
def load_and_split_data(
    test_size: float,
    random_state: int,
    n_samples: int,
    train_dataset: Output[Dataset],
    test_dataset: Output[Dataset],
):
    """Fetch the Covertype dataset, subsample, and split into train/test."""
    import pandas as pd
    from sklearn.datasets import fetch_covtype
    from sklearn.model_selection import train_test_split

    # Fetch full dataset (~580k rows) and subsample for speed
    data = fetch_covtype(as_frame=True)
    df = data.frame
    df = df.sample(n=min(n_samples, len(df)), random_state=random_state)

    # Split
    train_df, test_df = train_test_split(
        df, test_size=test_size, random_state=random_state, stratify=df['Cover_Type']
    )

    # Write to artifact paths — KFP handles GCS upload
    train_df.to_csv(train_dataset.path, index=False)
    test_df.to_csv(test_dataset.path, index=False)

    print(f"✅ Data loaded: {len(train_df)} train / {len(test_df)} test samples")

print("✅ load_and_split_data component defined.")

✅ load_and_split_data component defined.


### 2.2 Training component

Now it's your turn. Write a component that trains a **`RandomForestClassifier`** from scikit-learn.

The component should:
- Accept `train_dataset` as `Input[Dataset]` and hyperparameters as simple types (`int`, `float`)
- Output the trained model as `Output[Model]`
- Store training metadata (hyperparameters, framework) on the model artifact

> 📖 **Docs:**
> - [Artifact metadata](https://www.kubeflow.org/docs/components/pipelines/user-guides/data-handling/artifacts/) — use `artifact.metadata["key"] = value` to attach metadata to an artifact.
> - [`pickle.dump`](https://docs.python.org/3/library/pickle.html#pickle.dump)

In [ ]:
##############################  TODO  ##############################
# Complete the train_model component.
# 1. Read train_dataset.path as a CSV into a pandas DataFrame
# 2. Separate features (X) and target (y = "Cover_Type" column)
# 3. Create and fit a RandomForestClassifier with the given hyperparams
# 4. Pickle the model to model_artifact.path
# 5. Attach metadata: n_estimators, max_depth, framework='scikit-learn'

@dsl.component(
    base_image="python:3.10",
    packages_to_install=["scikit-learn==1.5.2", "pandas==2.2.3"],
)
def train_model(
    train_dataset: Input[Dataset],
    n_estimators: int,
    max_depth: int,
    random_state: int,
    model_artifact: Output[Model],
):
    """Train a RandomForestClassifier on the training set."""
    import pickle
    import pandas as pd
    from sklearn.ensemble import RandomForestClassifier

    train_df = pd.read_csv(train_dataset.path)  # TODO — read CSV from train_dataset.path
    X_train = train_df.drop(columns=["Cover_Type"])   # TODO — all columns except Cover_Type
    y_train = train_df["Cover_Type"]   # TODO — Cover_Type column

    clf = RandomForestClassifier(n_estimators=n_estimators,max_depth=max_depth,random_state=random_state)  # TODO — create and fit RandomForestClassifier
    clf.fit(X_train, y_train)
    # TODO — save the model using pickle to model_artifact.path
    with open(model_artifact.path, "wb") as f: # TODO
        pickle.dump(clf, f)# TODO



    # TODO — attach metadata to model_artifact
    model_artifact.metadata["n_estimators"] = n_estimators
    model_artifact.metadata["max_depth"] = max_depth
    model_artifact.metadata["framework"] = "scikit-learn"

    print(f"✅ Model trained: n_estimators={n_estimators}, max_depth={max_depth}")
####################################################################

print("✅ train_model component defined.")

✅ train_model component defined.


### 2.3 Evaluation component

Write a component that evaluates the trained model on the test set. It should:
- Load the test data and the pickled model
- Compute **accuracy** and **weighted F1 score**
- Log both metrics to the `Metrics` artifact

> 📖 **Docs:**
> - [`Metrics` artifact](https://www.kubeflow.org/docs/components/pipelines/user-guides/data-handling/artifacts/#metrics) — use `metrics.log_metric(name, value)` to record evaluation metrics.
> - [`pickle.load`](https://docs.python.org/3/library/pickle.html#pickle.load)

In [ ]:
##############################  TODO  ##############################
# Complete the evaluate_model component.
# 1. Read test_dataset.path as a CSV
# 2. Load the model from model_artifact.path using pickle
# 3. Compute accuracy and weighted F1 score
# 4. Log both metrics using metrics.log_metric("name", value)

@dsl.component(
    base_image="python:3.10",
    packages_to_install=["scikit-learn==1.5.2", "pandas==2.2.3"],
)
def evaluate_model(
    test_dataset: Input[Dataset],
    model_artifact: Input[Model],
    metrics: Output[Metrics],
):
    """Evaluate the trained model and log metrics."""
    import pickle
    import pandas as pd
    from sklearn.metrics import accuracy_score, f1_score

    test_df = pd.read_csv(test_dataset.path)   # TODO — read CSV from test_dataset.path
    X_test = test_df.drop(columns=["Cover_Type"])    # TODO — features
    y_test = test_df["Cover_Type"]    # TODO — target

    # TODO — load the pickled model
    with open(model_artifact.path, "rb") as f:
      model = pickle.load(f) # TODO

    y_pred = model.predict(X_test)  # TODO — predict on X_test

    accuracy = accuracy_score(y_test, y_pred)  # TODO — compute accuracy
    f1 = f1_score(y_test, y_pred, average="weighted")        # TODO — compute weighted F1

    # TODO — log both metrics
    metrics.log_metric("accuracy",accuracy)  # TODO
    metrics.log_metric("f1",f1)  # TODO

    print(f"✅ Evaluation — accuracy: {accuracy:.4f}, F1: {f1:.4f}")
####################################################################

print("✅ evaluate_model component defined.")

✅ evaluate_model component defined.


**✏️ Question 1 — Component isolation**

a) Why must all imports be placed **inside** the component function body, rather than at the top of the notebook?

b) A teammate suggests putting data loading, training, and evaluation in a **single** component. What are the drawbacks of this approach from an MLOps perspective?

---
*Your answer:*

**a)** All imports must be inside the component function because each component runs separately in its own container. This means it does not use the notebook environment directly. If the imports are only at the top of the notebook, the component may not find them when it runs.

**b)** Putting data loading, training, and evaluation in one component is not a good idea because it makes the pipeline less flexible. It becomes harder to test, debug, and reuse parts of the workflow. It also reduces the benefits of pipelines, such as running only one step again instead of the whole process.

---

---
## 3 · Compose & Compile the Pipeline

Now that the components are defined, we wire them together into a **pipeline**. The `@dsl.pipeline` decorator marks a function as a pipeline definition. Inside, you instantiate components as tasks and pass outputs from one task as inputs to the next.

> 📖 **Docs:** [`@dsl.pipeline` decorator](https://www.kubeflow.org/docs/components/pipelines/user-guides/core-functions/compile-a-pipeline/)

### 3.1 Define the pipeline

The pipeline function receives **parameters** (hyperparameters, configuration) as arguments. These become the knobs you can turn when re-running the pipeline without changing the code.

In [ ]:
##############################  TODO  ##############################
# Complete the pipeline definition.
# 1. Call load_and_split_data with the pipeline parameters.
# 2. Call train_model, wiring the train_dataset output from step 1.
# 3. Call evaluate_model, wiring the test_dataset from step 1
#    and the model_artifact from step 2.
#
# Hint: task.outputs["output_name"] gives you a reference to a
#        task's output that you can pass as input to the next task.

@dsl.pipeline(
    name="covertype-training-pipeline-mregaieg",
    description="Train and evaluate a RandomForest on the Covertype dataset.",
)
def covertype_pipeline(
    n_estimators: int = 100,
    max_depth: int = 10,
    test_size: float = 0.2,
    n_samples: int = 20000,
    random_state: int = 42,
):

    # Step 1: Load and split data
    data_task = load_and_split_data(
        test_size=test_size,
        random_state=random_state,
        n_samples=n_samples
    )  # TODO

    # Step 2: Train model
    train_task = train_model(
        train_dataset=data_task.outputs["train_dataset"],
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state,
    )  # TODO

    # Step 3: Evaluate model
    eval_task =  evaluate_model(
        test_dataset=data_task.outputs["test_dataset"],
        model_artifact=train_task.outputs["model_artifact"],
    ) # TODO
####################################################################

print("✅ Pipeline defined.")

✅ Pipeline defined.


### 3.2 Compile the pipeline

Compilation converts the Python pipeline definition into a **YAML** specification that Vertex AI Pipelines can execute. This YAML is portable — you can share it, version-control it, and schedule it.

> 📖 **Docs:** [`compiler.Compiler().compile()`](https://kubeflow-pipelines.readthedocs.io/en/stable/source/compiler.html#kfp.compiler.Compiler.compile)

In [ ]:
##############################  TODO  ##############################
# Compile the pipeline to a YAML file.
# Use compiler.Compiler().compile() with:
#   - pipeline_func: your pipeline function
#   - package_path: "covertype_pipeline.yaml"

compiler.Compiler().compile(
    pipeline_func=covertype_pipeline,
    package_path="covertype_pipeline.yaml",
)  # TODO
####################################################################

print("✅ Pipeline compiled to covertype_pipeline.yaml")

✅ Pipeline compiled to covertype_pipeline.yaml


In [ ]:
# Inspect the some lines of the compiled YAML
with open("covertype_pipeline.yaml", "r") as f:
    for i, line in enumerate(f):
        if i < 40:
            print(line, end='')
        elif i < 195:
            continue
        elif i < 221:
            print(line, end='')
        else:
            break

# PIPELINE DEFINITION
# Name: covertype-training-pipeline-mregaieg
# Description: Train and evaluate a RandomForest on the Covertype dataset.
# Inputs:
#    max_depth: int [Default: 10.0]
#    n_estimators: int [Default: 100.0]
#    n_samples: int [Default: 20000.0]
#    random_state: int [Default: 42.0]
#    test_size: float [Default: 0.2]
components:
  comp-evaluate-model:
    executorLabel: exec-evaluate-model
    inputDefinitions:
      artifacts:
        model_artifact:
          artifactType:
            schemaTitle: system.Model
            schemaVersion: 0.0.1
        test_dataset:
          artifactType:
            schemaTitle: system.Dataset
            schemaVersion: 0.0.1
    outputDefinitions:
      artifacts:
        metrics:
          artifactType:
            schemaTitle: system.Metrics
            schemaVersion: 0.0.1
  comp-load-and-split-data:
    executorLabel: exec-load-and-split-data
    inputDefinitions:
      parameters:
        n_samples:
          parameterTy

**✏️ Question 2 — Pipeline compilation**

a) Why does KFP compile the pipeline to YAML instead of executing the Python function directly?

b) What information can you identify in the YAML file? (Look at the first 40 lines.)

---
*Your answer:*

**a)** KFP compiles the pipeline to YAML because the Python function is only used to describe the workflow. The YAML is a portable specification that Vertex AI or Kubeflow can read and execute step by step. This makes the pipeline easier to save, share, version, and run in another environment without depending on the notebook itself.

**b)** In the YAML file, it is possible to identify the pipeline name and description, the input parameters with their default values, the different components, and the inputs and outputs of each component. It also shows artifact types such as `Dataset`, `Model`, and `Metrics`, and the task dependencies inside the pipeline.

---

---
## 4 · Run on Vertex AI Pipelines

Now we submit the compiled pipeline to Vertex AI for execution. Each component will run in its own container on Google Cloud infrastructure.

> 📖 **Docs:** [`aiplatform.PipelineJob`](https://cloud.google.com/python/docs/reference/aiplatform/latest/google.cloud.aiplatform.PipelineJob)

### 4.1 Submit the pipeline

In [27]:
##############################  TODO  ##############################
# Create and submit a PipelineJob.
# Use aiplatform.PipelineJob() with:
#   - the pipeline display name
#   - the template path
#   - the pipelone root
#   - parameter_values: use the defaults (empty dict {} is fine)
#
# Then call job.run() to submit it.

job = aiplatform.PipelineJob(
    display_name=PIPELINE_DISPLAY_NAME,
    template_path="covertype_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={},
)  # TODO — create PipelineJob
job.run()        # TODO — run the job
####################################################################

print(f"✅ Pipeline completed: {job.display_name}")
print(f"   Job resource name: {job.resource_name}")

✅ Pipeline completed: lab04a-covertype-mregaieg
   Job resource name: projects/588027604890/locations/europe-west3/pipelineJobs/covertype-training-pipeline-mregaieg-20260310141800


### 4.2 Monitor execution

You can monitor the pipeline execution in the **Vertex AI console**.

> 💡 **Tip:** Go to the Vertex AI console → **Pipelines** section to see the DAG visualization, logs for each component, and artifact details.

**✏️ Question 3 — Pipeline execution**

a) In the Vertex AI console, examine the pipeline DAG. How does the execution order of the components match the data dependencies you defined?

b) What happens if the `train_model` component fails? Does the `evaluate_model` component still run? Why or why not?

---
*Your answer:*

**a)** From the pipeline DAG, it can be seen that the execution starts with **load-and-split-data**, then continues with **train-model**, and finally **evaluate-model**. This matches the data dependencies defined in the pipeline: the training step needs the training dataset produced by the first step, and the evaluation step needs both the test dataset from the first step and the trained model from the second step.

**b)** If the **train_model** component fails, the **evaluate_model** component will not run. This is because evaluate_model depends on the output model artifact produced by train_model. Since the required input would be missing, the pipeline stops and the dependent step is not executed.

---

---
## 5 · Parameterize & Re-run

A major advantage of pipelines over notebooks is **parameterization**: you can re-run the same pipeline with different configurations without touching any code. This is essential for hyperparameter tuning and reproducible experiments.

### 5.1 Run with different hyperparameters

In [28]:
##############################  TODO  ##############################
# Submit a second pipeline run with different hyperparameters.
# Try: n_estimators=200, max_depth=20 (the rest can stay default).
# Use a different display_name so you can distinguish the runs.

job_2 = aiplatform.PipelineJob(
    display_name=f"{PIPELINE_DISPLAY_NAME}-v2",
    template_path="covertype_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        "n_estimators": 200,  # TODO
        "max_depth": 20,     # TODO
    },
)
job_2.run()  # TODO — run the job
####################################################################

print(f"✅ Second pipeline completed: {job_2.display_name}")

✅ Second pipeline completed: lab04a-covertype-mregaieg-v2


### 5.2 Compare pipeline runs

You can list and compare pipeline runs programmatically.

In [31]:
# List recent pipeline jobs
jobs = aiplatform.PipelineJob.list(
    filter=f'display_name="{PIPELINE_DISPLAY_NAME}" OR display_name="{PIPELINE_DISPLAY_NAME}-v2"'
)

for j in jobs:
    params = dict(j.gca_resource.runtime_config.parameter_values)
    print(f"Run: {j.display_name:<40} State: {j.state.name:<20}")
    print(f"     Parameters: {params}")
    print()

Run: lab04a-covertype-mregaieg-v2             State: PIPELINE_STATE_SUCCEEDED
     Parameters: {'max_depth': 20.0, 'n_estimators': 200.0}

Run: lab04a-covertype-mregaieg                State: PIPELINE_STATE_SUCCEEDED
     Parameters: {}



> 💡 **Tip:** In the Vertex AI console → Pipelines, click on each run to compare the evaluation metrics logged by the `evaluate_model` component. You can see them in the **metrics** artifcat's details of each run.

**✏️ Question 4 — Parameterization & reproducibility**

a) Why are one of the pipelines parameters printed as an empty dict?

b) Why is it better to change hyperparameters through `parameter_values` rather than editing the component code and recompiling?

c) How would you integrate pipeline parameterization with the Vertex AI Experiments you used in Lab 02a?

---
*Your answer:*

**a)** One of the pipeline parameters is printed as an empty dict because, in the first pipeline job, the submission was done with `parameter_values={}`. This means no custom values were provided, so the pipeline used only its default parameter values. In the second job, `parameter_values` was filled with values such as `n_estimators=200` and `max_depth=20`, so those parameters were explicitly changed for that run.

**b)** It is better to change hyperparameters through `parameter_values` because it is faster and cleaner. The pipeline code stays the same, and only the run configuration changes. This improves reproducibility because each run keeps a clear record of which parameter values were used, and it avoids modifying the component code and recompiling the pipeline every time a new experiment is tested.

**c)** Pipeline parameterization can be integrated with Vertex AI Experiments by logging each pipeline run as an experiment run and recording the parameter values used for that execution. In this way, each run can be compared using its hyperparameters and evaluation metrics, which makes experiment tracking easier. For example, one run could use the default parameters and another could use `n_estimators=200` and `max_depth=20`, and both runs could then be compared directly inside Vertex AI Experiments.


---

---
## 6 · Cleanup

Delete the pipeline runs to avoid unnecessary storage costs.

In [32]:
# Delete both pipeline jobs
for j in [job, job_2]:
    try:
        j.delete()
        print(f"✅ Deleted: {j.display_name}")
    except Exception as e:
        print(f"⚠️ Could not delete {j.display_name}: {e}")

✅ Deleted: lab04a-covertype-mregaieg
✅ Deleted: lab04a-covertype-mregaieg-v2


---
## Summary

In this lab, you learned to:

| Step | What you did | Tool / Feature used |
|------|-------------|---------------------|
| Components | Built lightweight Python components with typed I/O | `@dsl.component`, `Input`, `Output`, `Dataset`, `Model`, `Metrics` |
| Pipeline | Composed components into a DAG | `@dsl.pipeline` |
| Compilation | Compiled the pipeline to a portable YAML file | `compiler.Compiler().compile()` |
| Execution | Submitted the pipeline to Vertex AI Pipelines | `aiplatform.PipelineJob` |
| Parameterization | Re-ran the pipeline with different hyperparameters | `parameter_values` |

**Next lab:** In Lab 04b, you will build **production pipeline patterns**: data validation gates, conditional deployment logic, and model registration — turning a simple pipeline into a production-grade ML workflow.